# College Canteen Sales & Customer Spending Analysis

**Team:** Abel Joseph & Abhimanue Jijish (25BCAR0055)

## Objective
Analyze college canteen sales data to identify popular food items, customer spending patterns, peak sales periods, and revenue trends using Python.

## Workflow
Problem Definition → Data Collection → Data Cleaning → Data Processing → Data Analysis → Visualization → Findings → Conclusion

## 1. Problem Definition

A college canteen generates many orders every day, but raw sales records are difficult to interpret directly. This project processes the order data and uses statistical analysis and visualizations to understand sales patterns.

**Objectives:**
- Identify the most popular food and beverage items.
- Find the categories generating the most revenue.
- Analyze peak ordering periods and daily sales.
- Study customer spending patterns.
- Present the findings using clear visualizations.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Libraries imported successfully.')

## 2. Data Collection

The dataset is a **synthetic college canteen sales dataset** created for this mini project. It contains realistic order records and a few intentionally imperfect records so that the data-cleaning process can be demonstrated.

In [ ]:
# Change this path if the CSV is stored somewhere else.
FILE_PATH = 'college_canteen_sales_raw.csv'

df = pd.read_csv(FILE_PATH)

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Basic dataset inspection
print('Column information:')
print(df.info())

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

## 3. Data Cleaning

The raw dataset contains a few missing values, inconsistent text formatting, and a duplicate row. These issues are cleaned before analysis.

In [ ]:
# Remove exact duplicate records
df = df.drop_duplicates().copy()

# Standardize text columns
df['Item'] = df['Item'].astype('string').str.strip().str.title()
df['Category'] = df['Category'].astype('string').str.strip().str.title()
df['Payment_Method'] = df['Payment_Method'].astype('string').str.strip().str.upper()

# Fill missing category/payment values using the most frequent value
df['Category'] = df['Category'].fillna(df['Category'].mode()[0])
df['Payment_Method'] = df['Payment_Method'].fillna(df['Payment_Method'].mode()[0])

# Fill missing quantity with the median quantity
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median()).astype(int)

# Convert dates and calculate total amount again after cleaning
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Total_Amount'] = df['Quantity'] * df['Unit_Price']

print('Cleaning completed.')
print('Remaining missing values:')
print(df.isnull().sum())

## 4. Regular Expressions

Regular expressions are used to validate the format of order IDs. A valid ID follows the pattern `ORD` followed by exactly four digits.

In [ ]:
order_id_pattern = r'^ORD\d{4}$'

df['Valid_Order_ID'] = df['Order_ID'].str.match(order_id_pattern, na=False)

invalid_orders = df.loc[~df['Valid_Order_ID'], ['Order_ID']]

print('Invalid order IDs found:', len(invalid_orders))
print(invalid_orders if len(invalid_orders) else 'None')

**Why Regex is useful here:** it checks that every order identifier follows the required structure instead of relying only on visual inspection.

## 5. Data Structures

Python data structures are used to store and summarize information before and during analysis.

In [ ]:
# List: store all unique food items
food_items = df['Item'].unique().tolist()

# Dictionary: map each item to its category
item_category = dict(zip(df['Item'], df['Category']))

# Tuple: store important summary values as one fixed collection
summary_tuple = (len(df), df['Quantity'].sum(), df['Total_Amount'].sum())

print('Number of unique items:', len(food_items))
print('First 5 items:', food_items[:5])
print('Example item-category mapping:', list(item_category.items())[:5])
print('Summary tuple (orders, units sold, revenue):', summary_tuple)

## 6. Object-Oriented Programming

An `Order` class represents an individual canteen order. This demonstrates how OOP can model real-world entities.

In [ ]:
class Order:
    def __init__(self, order_id, item, quantity, unit_price):
        self.order_id = order_id
        self.item = item
        self.quantity = quantity
        self.unit_price = unit_price

    def total_cost(self):
        return self.quantity * self.unit_price

    def display(self):
        return f'{self.order_id}: {self.quantity} x {self.item} = ₹{self.total_cost():.2f}'


sample_order = Order('ORD0001', 'Coffee', 2, 25)
print(sample_order.display())

## 7. NumPy Analysis

NumPy is used for numerical calculations such as mean, median, minimum, maximum, and standard deviation.

In [ ]:
amounts = df['Total_Amount'].to_numpy()
quantities = df['Quantity'].to_numpy()

print(f'Average order amount: ₹{np.mean(amounts):.2f}')
print(f'Median order amount: ₹{np.median(amounts):.2f}')
print(f'Minimum order amount: ₹{np.min(amounts):.2f}')
print(f'Maximum order amount: ₹{np.max(amounts):.2f}')
print(f'Standard deviation of order amount: ₹{np.std(amounts):.2f}')
print(f'Total units sold: {np.sum(quantities)}')

## 8. Pandas Analysis

In [ ]:
# Add useful time features
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
df['Day'] = df['Date'].dt.day_name()

item_sales = (df.groupby('Item', as_index=False)
                .agg(Units_Sold=('Quantity', 'sum'), Revenue=('Total_Amount', 'sum'))
                .sort_values('Units_Sold', ascending=False))

category_sales = (df.groupby('Category', as_index=False)
                    .agg(Orders=('Order_ID', 'count'), Units_Sold=('Quantity', 'sum'), Revenue=('Total_Amount', 'sum'))
                    .sort_values('Revenue', ascending=False))

daily_sales = (df.groupby('Date', as_index=False)
                 .agg(Orders=('Order_ID', 'count'), Revenue=('Total_Amount', 'sum'))
                 .sort_values('Date'))

print('Top 10 items by units sold:')
display(item_sales.head(10))

print('Category performance:')
display(category_sales)

In [ ]:
# Peak ordering hour
hourly_sales = (df.groupby('Hour')
                  .agg(Orders=('Order_ID', 'count'), Revenue=('Total_Amount', 'sum'))
                  .sort_values('Orders', ascending=False))

peak_hour = hourly_sales.index[0]

print(f'Peak ordering hour: {peak_hour}:00')
display(hourly_sales.head(10))

# Payment method distribution
payment_summary = df.groupby('Payment_Method').agg(
    Orders=('Order_ID', 'count'),
    Revenue=('Total_Amount', 'sum')
).sort_values('Orders', ascending=False)

display(payment_summary)

## 9. Visualizations

Each visualization is followed by a short interpretation of the insight it provides.

In [ ]:
# Visualization 1: Top 10 items by units sold
top_items = item_sales.head(10).sort_values('Units_Sold')

plt.figure(figsize=(10, 6))
plt.barh(top_items['Item'], top_items['Units_Sold'])
plt.xlabel('Units Sold')
plt.ylabel('Food Item')
plt.title('Top 10 Most Sold Canteen Items')
plt.tight_layout()
plt.show()

print('Insight: This chart shows which individual items contribute the most to the number of units sold.')

In [ ]:
# Visualization 2: Revenue by category
plt.figure(figsize=(9, 5))
plt.bar(category_sales['Category'], category_sales['Revenue'])
plt.xlabel('Category')
plt.ylabel('Revenue (₹)')
plt.title('Revenue by Food Category')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('Insight: The category with the tallest bar contributes the highest total revenue.')

In [ ]:
# Visualization 3: Daily revenue trend
plt.figure(figsize=(11, 5))
plt.plot(daily_sales['Date'], daily_sales['Revenue'], marker='o')
plt.xlabel('Date')
plt.ylabel('Revenue (₹)')
plt.title('Daily Canteen Revenue Trend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Insight: The line shows how canteen revenue changes from day to day and highlights unusually high or low sales days.')

In [ ]:
# Visualization 4: Orders by hour
hourly_plot = hourly_sales.sort_index()

plt.figure(figsize=(10, 5))
plt.bar(hourly_plot.index.astype(str), hourly_plot['Orders'])
plt.xlabel('Hour of Day')
plt.ylabel('Number of Orders')
plt.title('Orders by Hour')
plt.tight_layout()
plt.show()

print('Insight: The chart identifies the busiest ordering hours, which can help with staffing and food preparation.')

In [ ]:
# Visualization 5: Customer spending distribution
plt.figure(figsize=(9, 5))
plt.hist(df['Total_Amount'], bins=12)
plt.xlabel('Order Amount (₹)')
plt.ylabel('Number of Orders')
plt.title('Distribution of Customer Order Spending')
plt.tight_layout()
plt.show()

print('Insight: The histogram shows the most common spending range and how widely order values vary.')

## 10. Key Findings

The following values are calculated automatically from the cleaned dataset so that the findings remain consistent with the analysis.

In [ ]:
top_item = item_sales.iloc[0]
top_category = category_sales.iloc[0]
highest_day = daily_sales.loc[daily_sales['Revenue'].idxmax()]
average_spend = df['Total_Amount'].mean()
total_revenue = df['Total_Amount'].sum()
total_orders = len(df)

print(f'1. Total cleaned orders: {total_orders}')
print(f'2. Total revenue: ₹{total_revenue:,.2f}')
print(f'3. Average order value: ₹{average_spend:,.2f}')
print(f'4. Most sold item: {top_item["Item"]} ({int(top_item["Units_Sold"])} units)')
print(f'5. Highest-revenue category: {top_category["Category"]} (₹{top_category["Revenue"]:,.2f})')
print(f'6. Highest-revenue date: {highest_day["Date"].date()} (₹{highest_day["Revenue"]:,.2f})')
print(f'7. Peak ordering hour: {peak_hour}:00')

## 11. Conclusion

The project demonstrates how Python can transform raw canteen transaction records into useful information. Pandas was used for data cleaning and grouping, NumPy for numerical analysis, regular expressions for validation, Python data structures for organizing information, and OOP to represent an order as a real-world object. The visualizations make sales trends, popular items, peak periods, and spending patterns easier to understand.

The results can help a canteen understand demand patterns and make more informed decisions about food preparation, stock planning, and staffing.

## 12. Python Concepts Used

| Requirement | Where it is used |
|---|---|
| Python Programming | Variables, functions, loops, conditions and calculations throughout the notebook |
| Data Structures | List, dictionary and tuple in the data-structure section |
| Object-Oriented Programming | `Order` class and object |
| Regular Expressions | Order ID validation using `re` |
| NumPy | Mean, median, minimum, maximum, standard deviation and sum |
| Pandas | Data loading, cleaning, grouping, filtering and feature creation |
| Visualization | Matplotlib charts for sales, revenue, time and spending |
| Advanced concept | Object-oriented design and feature engineering from date/time data |

## Dataset Source Note

The dataset used in this mini project is synthetic and was generated specifically for educational analysis. No real student/customer personal information is included.